In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
genres_df = spark.read.format("delta").load(f"{silver_folder_path}/movie_genres")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()

In [0]:
from pyspark.sql import functions as F

final_movies_df = (
  ratings_df
    .join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .groupBy(movies_metadata_df.id, "title", "overview")
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter("number_of_ratings > 100")
    .orderBy(F.col("average_rating").desc())
)

display(final_movies_df)

In [0]:
import plotly.express as px

top20_pdf = (
  final_movies_df
    .orderBy(F.col("average_rating").desc())
    .limit(20)
    .toPandas()
)

fig_top20 = px.bar(
  top20_pdf,
  x="average_rating",
  y="title",
  orientation="h",
  hover_data=["number_of_ratings", "id"],
  title="Top 20 movies by average rating (≥100 ratings)",
  labels={"average_rating": "Average rating", "title": "Movie"},
  text=top20_pdf["average_rating"].round(2),
  range_x=[4.0, 4.6],
)
fig_top20.update_traces(textposition="outside", cliponaxis=False)
fig_top20.update_layout(yaxis={"categoryorder": "total ascending"}, height=650, margin=dict(r=80))
fig_top20.show()

In [0]:
scatter_pdf = (
  final_movies_df
    .orderBy(F.col("number_of_ratings").desc())
    .limit(200)
    .toPandas()
)

fig_scatter = px.scatter(
  scatter_pdf,
  x="number_of_ratings",
  y="average_rating",
  hover_name="title",
  hover_data=["id"],
  title="Popularity vs quality (top 200 by rating count)",
  labels={
    "number_of_ratings": "Number of ratings",
    "average_rating": "Average rating",
  },
  log_x=True,
)
fig_scatter.show()